# LeetCode 1175 — Prime Arrangements bằng Python

**Link đề:** https://leetcode.com/problems/prime-arrangements/
**Độ khó:** Easy · **Chủ đề:** Math, Primality Test, Sieve, Combinatorics

```python
class Solution:
    def numPrimeArrangements(self, n: int) -> int:
        ...
```

## Mục tiêu học
1. Đọc đề tổ hợp: "đếm số **hoán vị** thỏa điều kiện" và nhận ra bài **tách được thành hai nhóm độc lập**.
2. Kiểm tra số nguyên tố bằng **thử chia tới √x**, rồi bằng **sàng Eratosthenes**.
3. Nguyên lý nhân: đáp án = `p! × (n − p)!` với `p` = số số nguyên tố trong `1..n`.
4. Tính **giai thừa modulo `10^9 + 7`** đúng cách (lấy modulo ở mỗi bước).
5. Kiểm chứng bằng **brute force** liệt kê hoán vị với `n` nhỏ.
6. Trace, chứng minh, test, lỗi hay gặp, biến thể, bài tập.

## 1. Đề bài và ví dụ

Trả về số **hoán vị** của các số `1..n` sao cho **các số nguyên tố nằm ở các vị trí nguyên tố** (vị trí đánh số từ **1**).

Nhắc lại: một số nguyên là **nguyên tố** nếu nó `> 1` và không viết được thành tích của hai số nguyên dương đều nhỏ hơn nó.

Vì đáp án có thể rất lớn, trả về **modulo `10^9 + 7`**.

| `n` | Output | Giải thích |
|---:|---:|---|
| `5` | `12` | ví dụ `[1,2,5,4,3]` hợp lệ; `[5,2,3,4,1]` không hợp lệ vì số nguyên tố 5 ở vị trí 1 |
| `100` | `682289015` | |

**Constraints:** `1 <= n <= 100`.

In [1]:
def is_valid_arrangement(perm):
    # perm là list, vị trí đánh số từ 1
    for idx, value in enumerate(perm, start=1):
        if is_prime_simple(value) != is_prime_simple(idx):
            return False
    return True

def is_prime_simple(x):
    return x > 1 and all(x % d != 0 for d in range(2, x))

print(is_valid_arrangement([1, 2, 5, 4, 3]))   # True
print(is_valid_arrangement([5, 2, 3, 4, 1]))   # False

True
False


### Đọc kỹ điều kiện

"Số nguyên tố ở vị trí nguyên tố" — vì có **đúng `p` số nguyên tố** và **đúng `p` vị trí nguyên tố** trong `1..n`, nên khi mọi số nguyên tố đã chiếm hết các vị trí nguyên tố thì các số **không nguyên tố** buộc phải nằm ở vị trí **không nguyên tố**. Hai nhóm hoàn toàn tách biệt.

Với `n = 5`: số nguyên tố `{2, 3, 5}` (p = 3), không nguyên tố `{1, 4}`.
- Xếp `{2,3,5}` vào vị trí `{2,3,5}`: `3! = 6` cách.
- Xếp `{1,4}` vào vị trí `{1,4}`: `2! = 2` cách.
- Tổng: `6 × 2 = 12` ✔.

## 2. Nền tảng cần biết

### 2.1. Hoán vị và giai thừa

Số cách xếp `k` đồ vật khác nhau vào `k` chỗ là `k! = 1 × 2 × ... × k` (quy ước `0! = 1`).
`itertools.permutations(lst)` liệt kê **mọi** hoán vị — dùng để kiểm tra với `k` nhỏ.

In [2]:
from itertools import permutations
import math

print(list(permutations([1, 2, 3])))
print(len(list(permutations([1, 2, 3]))), math.factorial(3))
print("0! =", math.factorial(0))

[(1, 2, 3), (1, 3, 2), (2, 1, 3), (2, 3, 1), (3, 1, 2), (3, 2, 1)]
6 6
0! = 1


### 2.2. Nguyên lý nhân (rule of product)

Nếu việc A có `a` cách, việc B có `b` cách và **hai việc độc lập**, thì làm cả hai có `a × b` cách. Ví dụ 3 áo × 2 quần = 6 bộ.

### 2.3. Phép modulo `%` và hằng `10**9 + 7`

- `x % m` là phần dư khi chia `x` cho `m`.
- Tính chất quan trọng: `(a × b) % m = ((a % m) × (b % m)) % m`. Nhờ đó ta lấy modulo **sau mỗi phép nhân** để số không phình to.
- `10**9 + 7` là số nguyên tố lớn thường dùng trong đề thi.

In [3]:
MOD = 10**9 + 7
a, b_ = 123456789123, 987654321987
print((a * b_) % MOD)
print(((a % MOD) * (b_ % MOD)) % MOD)   # giống nhau

187830321
187830321


### 2.4. Kiểm tra nguyên tố bằng thử chia tới √x

Nếu `x = d × e` với `d <= e` thì `d <= √x`. Vậy chỉ cần thử các ước `d` từ 2 tới `⌊√x⌋`. `math.isqrt(x)` trả căn bậc hai **nguyên** (làm tròn xuống), tránh sai số số thực.

In [4]:
def is_prime(x: int) -> bool:
    if x < 2:
        return False
    for d in range(2, math.isqrt(x) + 1):
        if x % d == 0:
            return False
    return True

print(math.isqrt(10), math.isqrt(16), math.isqrt(17))
print([x for x in range(1, 31) if is_prime(x)])

3 4 4
[2, 3, 5, 7, 11, 13, 17, 19, 23, 29]


Lưu ý `range(2, isqrt(x) + 1)` — phải `+ 1` để thử cả `⌊√x⌋` (ví dụ `x = 25` cần thử `d = 5`).

### 2.5. Sàng Eratosthenes

Đánh dấu mọi số `2..n` là "có thể nguyên tố". Với mỗi số nguyên tố `i` (từ 2 tới √n), gạch bỏ các bội `i*i, i*i + i, ...`. Số còn lại là nguyên tố. Tìm **tất cả** số nguyên tố `<= n` trong `O(n log log n)`.

In [5]:
def sieve(n: int) -> list:
    is_p = [True] * (n + 1)
    is_p[0] = False
    if n >= 1:
        is_p[1] = False
    i = 2
    while i * i <= n:
        if is_p[i]:
            for k in range(i * i, n + 1, i):   # bước nhảy i
                is_p[k] = False
        i += 1
    return is_p

flags = sieve(30)
print([x for x in range(31) if flags[x]])
print(list(range(9, 31, 3)))   # range(start, stop, step)

[2, 3, 5, 7, 11, 13, 17, 19, 23, 29]
[9, 12, 15, 18, 21, 24, 27, 30]


Vì sao bắt đầu gạch từ `i * i`? Các bội nhỏ hơn như `2i, 3i, ...` đã bị gạch bởi số nguyên tố nhỏ hơn rồi.

## 3. Cách 1 — Brute force: liệt kê mọi hoán vị (chỉ để hiểu)

Duyệt toàn bộ `n!` hoán vị, đếm hoán vị hợp lệ. Đúng tuyệt đối nhưng `n!` tăng khủng khiếp: `10! = 3 628 800`, `100!` có 158 chữ số.

In [6]:
def prime_arrangements_brute(n: int) -> int:
    count = 0
    for perm in permutations(range(1, n + 1)):
        ok = True
        for idx, value in enumerate(perm, start=1):
            if is_prime(value) and not is_prime(idx):
                ok = False
                break
        if ok:
            count += 1
    return count

for n in range(1, 9):
    print(n, prime_arrangements_brute(n))

1 1
2 1
3 2
4 4
5 12
6 36
7 144
8 576


Chỉ cần kiểm tra "số nguyên tố ở vị trí không nguyên tố" là đủ: như đã lập luận, nếu mọi số nguyên tố đều ở vị trí nguyên tố thì số không nguyên tố tự động ở vị trí không nguyên tố.

### Nút thắt
`n!` hoán vị → không thể với `n = 100`. Cần **đếm mà không liệt kê**.

## 4. Insight — Tách hai nhóm, nhân giai thừa

```text
p = số lượng số nguyên tố trong 1..n
đáp án = p! × (n − p)!   (mod 10^9 + 7)
```

Bài quy về hai việc nhỏ: **đếm số nguyên tố** và **tính giai thừa modulo**.

In [7]:
for n in [5, 8]:
    p = sum(1 for x in range(1, n + 1) if is_prime(x))
    formula = math.factorial(p) * math.factorial(n - p)
    print(f"n={n}: p={p}, {p}! × {n - p}! = {formula}, brute = {prime_arrangements_brute(n)}")

n=5: p=3, 3! × 2! = 12, brute = 12
n=8: p=4, 4! × 4! = 576, brute = 576


## 5. Cách 2 — Lời giải chính: đếm nguyên tố + giai thừa modulo

**Mổ xẻ:**
1. `p = sum(1 for x in range(2, n+1) if is_prime(x))`.
2. Hàm `fact_mod(k)`: `res = 1`, nhân dần `1..k`, mỗi bước `% MOD`.
3. Trả `fact_mod(p) * fact_mod(n - p) % MOD`.

In [8]:
MOD = 10**9 + 7

def fact_mod(k: int) -> int:
    res = 1
    for i in range(2, k + 1):
        res = res * i % MOD
    return res

def prime_arrangements(n: int) -> int:
    p = sum(1 for x in range(2, n + 1) if is_prime(x))
    return fact_mod(p) * fact_mod(n - p) % MOD

print(prime_arrangements(5), prime_arrangements(100))

12 682289015


**Thứ tự ưu tiên toán tử:** `res * i % MOD` được tính là `(res * i) % MOD` vì `*` và `%` cùng mức, tính từ trái sang phải. Viết thêm ngoặc cũng được nếu muốn rõ.

## 6. TRACE từng bước với `n = 10`

**Bước A — đếm nguyên tố:**

| x | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 |
|---|---|---|---|---|---|---|---|---|---|
| nguyên tố? | ✔ | ✔ | | ✔ | | ✔ | | | |

→ `p = 4`, `n - p = 6`.

**Bước B — giai thừa:** `4! = 24`, `6! = 720` → `24 × 720 = 17280`.

In [9]:
def trace_prime_arrangements(n: int) -> int:
    primes = []
    for x in range(1, n + 1):
        flag = is_prime(x)
        if flag:
            primes.append(x)
        print(f"  x={x:>3}: {'nguyên tố' if flag else '-'}")
    p = len(primes)
    print(f"p = {p} (các số {primes}), n - p = {n - p}")

    def trace_fact(k, name):
        res = 1
        for i in range(2, k + 1):
            res = res * i % MOD
            print(f"    {name}: nhân {i:>2} -> {res}")
        return res

    a = trace_fact(p, f"{p}!")
    c = trace_fact(n - p, f"{n - p}!")
    ans = a * c % MOD
    print(f"đáp án = {a} × {c} mod MOD = {ans}")
    return ans

trace_prime_arrangements(10)

  x=  1: -
  x=  2: nguyên tố
  x=  3: nguyên tố
  x=  4: -
  x=  5: nguyên tố
  x=  6: -
  x=  7: nguyên tố
  x=  8: -
  x=  9: -
  x= 10: -
p = 4 (các số [2, 3, 5, 7]), n - p = 6
    4!: nhân  2 -> 2
    4!: nhân  3 -> 6
    4!: nhân  4 -> 24
    6!: nhân  2 -> 2
    6!: nhân  3 -> 6
    6!: nhân  4 -> 24
    6!: nhân  5 -> 120
    6!: nhân  6 -> 720
đáp án = 24 × 720 mod MOD = 17280


17280

## 7. Cách 3 — Dùng sàng + `math.factorial`

Python có số nguyên lớn không giới hạn, nên có thể tính `math.factorial` trực tiếp rồi mới `% MOD`. Với `n <= 100` vẫn rất nhanh. Còn ở C++/Java, phải lấy modulo từng bước để tránh tràn số.

In [10]:
def prime_arrangements_sieve(n: int) -> int:
    flags = sieve(n)
    p = sum(flags)                 # True được tính là 1, False là 0
    return math.factorial(p) * math.factorial(n - p) % MOD

print(sum([True, False, True]))   # 2
print(prime_arrangements_sieve(5), prime_arrangements_sieve(100))
print("100! có", len(str(math.factorial(100))), "chữ số")

2
12 682289015
100! có 158 chữ số


## 8. Cách 4 — Bảng 25 số nguyên tố nhỏ hơn 100 (hằng số)

Vì `n <= 100`, có thể ghi cứng danh sách 25 số nguyên tố `< 100` và đếm bằng `bisect_right`. Đây là "mẹo thi đấu", không cần thuật toán kiểm tra nguyên tố.

In [11]:
from bisect import bisect_right

PRIMES_100 = [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47,
              53, 59, 61, 67, 71, 73, 79, 83, 89, 97]

def prime_arrangements_table(n: int) -> int:
    p = bisect_right(PRIMES_100, n)       # số phần tử <= n
    return math.factorial(p) * math.factorial(n - p) % MOD

print(len(PRIMES_100), prime_arrangements_table(5), prime_arrangements_table(100))
assert PRIMES_100 == [x for x in range(101) if sieve(100)[x]]

25 12 682289015


## 9. Code nộp LeetCode

In [12]:
import math

class Solution:
    def numPrimeArrangements(self, n: int) -> int:
        MOD = 10**9 + 7

        def is_prime(x: int) -> bool:
            if x < 2:
                return False
            for d in range(2, math.isqrt(x) + 1):
                if x % d == 0:
                    return False
            return True

        p = sum(1 for x in range(2, n + 1) if is_prime(x))
        res = 1
        for i in range(2, p + 1):
            res = res * i % MOD
        for i in range(2, n - p + 1):
            res = res * i % MOD
        return res

sol = Solution()
print(sol.numPrimeArrangements(5), sol.numPrimeArrangements(100))

12 682289015


In [13]:
class SolutionSieve:
    def numPrimeArrangements(self, n: int) -> int:
        is_p = [False, False] + [True] * (n - 1)     # chỉ số 0..n
        for i in range(2, math.isqrt(n) + 1):
            if is_p[i]:
                for k in range(i * i, n + 1, i):
                    is_p[k] = False
        p = sum(is_p[: n + 1])
        return math.factorial(p) * math.factorial(n - p) % (10**9 + 7)

print(SolutionSieve().numPrimeArrangements(1), SolutionSieve().numPrimeArrangements(100))

1 682289015


Kiểm tra độ dài `is_p` khi `n = 1`: `[False, False] + [True] * 0` dài 2 — chỉ số 0, 1 đều có, không lỗi.

## 10. Vì sao đúng?

1. Gọi `P` = tập số nguyên tố trong `1..n`, `|P| = p`. Tập **vị trí** nguyên tố cũng chính là `P` (cùng miền `1..n`), nên có đúng `p` vị trí nguyên tố.
2. Hoán vị hợp lệ ⇔ ánh xạ `P → P` (số nguyên tố vào vị trí nguyên tố). Khi đó `n - p` số còn lại phải vào `n - p` vị trí còn lại.
3. Cách xếp nhóm nguyên tố: `p!`; nhóm còn lại: `(n − p)!`; hai lựa chọn độc lập → nhân lại (nguyên lý nhân).
4. Lấy modulo từng bước đúng nhờ tính chất `(a·b) mod m = ((a mod m)(b mod m)) mod m`.

Brute force với `n <= 8` khớp công thức — kiểm chứng thực nghiệm cho lập luận.

## 11. Độ phức tạp

| Cách | Time | Space |
|---|---|---|
| Brute force hoán vị | `O(n! · n)` | `O(n)` |
| Thử chia + giai thừa | `O(n√n)` đếm nguyên tố + `O(n)` giai thừa | `O(1)` |
| Sàng + giai thừa | `O(n log log n) + O(n)` | `O(n)` |
| Bảng 25 số | `O(log 25) + O(n)` | `O(1)` |

Với `n <= 100` mọi cách (trừ brute force) đều tức thì.

## 12. Bộ test: mẫu, biên, đối chiếu brute force

In [14]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<12} PASS ({actual})")

sol = Solution()
check("đề n=5", sol.numPrimeArrangements(5), 12)
check("đề n=100", sol.numPrimeArrangements(100), 682289015)
check("n=1", sol.numPrimeArrangements(1), 1)       # p=0: 0! × 1! = 1
check("n=2", sol.numPrimeArrangements(2), 1)       # p=1: 1! × 1! = 1
check("n=3", sol.numPrimeArrangements(3), 2)       # p=2: 2! × 1! = 2
check("n=4", sol.numPrimeArrangements(4), 4)       # p=2: 2! × 2! = 4

đề n=5       PASS (12)
đề n=100     PASS (682289015)
n=1          PASS (1)
n=2          PASS (1)
n=3          PASS (2)
n=4          PASS (4)


In [15]:
for n in range(1, 9):
    expected = prime_arrangements_brute(n)
    for f in (prime_arrangements, prime_arrangements_sieve, prime_arrangements_table,
              Solution().numPrimeArrangements, SolutionSieve().numPrimeArrangements):
        assert f(n) == expected, (f, n)
print("n = 1..8: mọi cách khớp brute force. PASS")

for n in range(1, 101):
    expected = prime_arrangements_sieve(n)
    assert prime_arrangements(n) == expected
    assert prime_arrangements_table(n) == expected
    assert Solution().numPrimeArrangements(n) == expected
print("n = 1..100: các cách nhanh khớp nhau. PASS")

n = 1..8: mọi cách khớp brute force. PASS
n = 1..100: các cách nhanh khớp nhau. PASS


## 13. Lỗi người mới hay gặp

### Lỗi 1: Coi 1 là số nguyên tố
```python
def is_prime(x): return all(x % d for d in range(2, x))   # x = 1 → True (SAI)
```
`all([])` là `True` nên 1 lọt qua. Phải kiểm tra `x < 2` trước.

### Lỗi 2: `range(2, isqrt(x))` thiếu `+ 1` → các số chính phương như 4, 9, 25, 49 và cả 6, 8, 15, 35… (ước nhỏ nhất đúng bằng `isqrt(x)`) bị coi là nguyên tố.

### Lỗi 3: Quên modulo hoặc chỉ modulo một vế
Ở Python vẫn ra đúng vì số lớn, nhưng nếu quên `% MOD` cuối cùng thì kết quả là con số khổng lồ.

### Lỗi 4: Dùng `p! × p!` hoặc `n!` — hiểu sai cấu trúc hai nhóm.

In [16]:
def wrong_is_prime_one(x):
    return all(x % d for d in range(2, x))

def wrong_is_prime_sqrt(x):
    if x < 2:
        return False
    return all(x % d for d in range(2, math.isqrt(x)))   # thiếu +1

print("1 nguyên tố?", wrong_is_prime_one(1), "| đúng:", is_prime(1))
print("số bị nhận nhầm:", [x for x in range(2, 50) if wrong_is_prime_sqrt(x) != is_prime(x)])

def wrong_no_mod(n):
    p = sum(1 for x in range(2, n + 1) if is_prime(x))
    return math.factorial(p) * math.factorial(n - p)

print("quên mod với n=100 có", len(str(wrong_no_mod(100))), "chữ số")

1 nguyên tố? True | đúng: False
số bị nhận nhầm: [4, 6, 8, 9, 15, 25, 35, 49]
quên mod với n=100 có 135 chữ số


## 14. Mẹo nhận dạng pattern

- "Đếm số hoán vị thỏa điều kiện" + điều kiện chia phần tử thành **nhóm độc lập** → **nhân giai thừa** từng nhóm.
- "Trả modulo `10^9 + 7`" → lấy `%` sau **mỗi** phép nhân/cộng.
- "Đếm số nguyên tố `<= n`" → thử chia tới `√x` (n nhỏ) hoặc **sàng Eratosthenes** (n lớn, bài 204 Count Primes).
- Nghi ngờ công thức? Viết **brute force** cho `n` nhỏ và so sánh.

## 15. Biến thể

### 15.1. Count Primes (204) — đếm số nguyên tố **nhỏ hơn** n bằng sàng

In [17]:
def count_primes(n: int) -> int:
    if n < 3:
        return 0
    flags = sieve(n - 1)
    return sum(flags)

print(count_primes(10), count_primes(0), count_primes(100))

4 0 25


### 15.2. Điều kiện khác: số chẵn ở vị trí chẵn
Cùng cấu trúc: số chẵn trong `1..n` có `n // 2` số, vị trí chẵn cũng `n // 2` → đáp án `(n//2)! × (n - n//2)!`.

In [18]:
def even_at_even(n: int) -> int:
    e = n // 2
    return math.factorial(e) * math.factorial(n - e) % MOD

def even_at_even_brute(n: int) -> int:
    return sum(1 for perm in permutations(range(1, n + 1))
               if all((v % 2 == 0) == (i % 2 == 0) for i, v in enumerate(perm, start=1)))

for n in range(1, 8):
    assert even_at_even(n) == even_at_even_brute(n)
print([even_at_even(n) for n in range(1, 8)])

[1, 1, 2, 4, 12, 36, 144]


### 15.3. So sánh các cách

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| Brute force | liệt kê hoán vị | `O(n!·n)` | `O(n)` | chỉ để kiểm chứng |
| Thử chia + giai thừa mod | đếm p, nhân dần | `O(n√n)` | `O(1)` | **nộp bài**, dễ chuyển sang ngôn ngữ khác |
| Sàng + factorial | đếm p bằng sàng | `O(n log log n)` | `O(n)` | tốt khi n lớn |
| Bảng hằng | 25 số nguyên tố | `O(n)` | `O(1)` | mẹo, chỉ hợp với n ≤ 100 |

## 16. Bài tập vận dụng (TODO)

### Bài 1 — Tính tay
Với `n = 7`: liệt kê số nguyên tố, tính `p`, rồi tính đáp án.

### Bài 2 — Viết `is_prime` thử chia tới √x

In [19]:
def is_prime_exercise(x: int) -> bool:
    # TODO: x < 2 → False; thử d từ 2 tới isqrt(x)
    pass

# assert [x for x in range(1, 20) if is_prime_exercise(x)] == [2, 3, 5, 7, 11, 13, 17, 19]

### Bài 3 — Giai thừa modulo

In [20]:
def fact_mod_exercise(k: int, mod: int = 10**9 + 7) -> int:
    # TODO: res = 1; nhân dần, lấy % mỗi bước
    pass

# assert fact_mod_exercise(0) == 1
# assert fact_mod_exercise(5) == 120
# assert fact_mod_exercise(100) == math.factorial(100) % (10**9 + 7)

### Bài 4 — Hoàn thiện lời giải bài chính

In [21]:
def prime_arrangements_exercise(n: int) -> int:
    # TODO: đếm p, trả fact(p) * fact(n - p) % MOD
    pass

# assert prime_arrangements_exercise(5) == 12
# assert prime_arrangements_exercise(100) == 682289015

### Bài 5 — Sàng trả về danh sách số nguyên tố `<= n`

In [22]:
def primes_up_to_exercise(n: int) -> list:
    # TODO: sàng Eratosthenes
    pass

# assert primes_up_to_exercise(1) == []
# assert primes_up_to_exercise(20) == [2, 3, 5, 7, 11, 13, 17, 19]

### Bài 6 — Mini challenge: ba nhóm
Đếm hoán vị của `1..n` sao cho số chia 3 dư `r` nằm ở vị trí chia 3 dư `r` (với mọi `r = 0, 1, 2`).

In [23]:
def mod3_arrangements_exercise(n: int) -> int:
    # TODO: đếm kích thước 3 nhóm, nhân 3 giai thừa
    pass

# assert mod3_arrangements_exercise(3) == 1
# assert mod3_arrangements_exercise(6) == 8

## 17. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | nguyên tố ≤ 7 | 2, 3, 5, 7 | `4! × 3!` |
| 2 | ước nhỏ ≤ √x | `math.isqrt` | `for d in range(2, isqrt(x) + 1)` |
| 3 | `0! = 1` | vòng `range(2, k + 1)` | `res = res * i % mod` |
| 4 | ghép bài 2 + 3 | `p = sum(...)` | `fact(p) * fact(n - p) % MOD` |
| 5 | gạch bội | bắt đầu từ `i * i` | `range(i*i, n+1, i)` |
| 6 | nhóm theo `x % 3` | số lượng mỗi nhóm số = số lượng mỗi nhóm vị trí | `Counter(x % 3 for x in range(1, n+1))` |

## 18. Đáp án tham khảo

**Bài 1:** nguyên tố `{2,3,5,7}` → `p = 4`, `n - p = 3` → `4! × 3! = 24 × 6 = ` **144**.

In [24]:
from collections import Counter

def is_prime_answer(x: int) -> bool:
    if x < 2:
        return False
    for d in range(2, math.isqrt(x) + 1):
        if x % d == 0:
            return False
    return True

def fact_mod_answer(k: int, mod: int = 10**9 + 7) -> int:
    res = 1
    for i in range(2, k + 1):
        res = res * i % mod
    return res

def prime_arrangements_answer(n: int) -> int:
    p = sum(1 for x in range(2, n + 1) if is_prime_answer(x))
    return fact_mod_answer(p) * fact_mod_answer(n - p) % MOD

def primes_up_to_answer(n: int) -> list:
    if n < 2:
        return []
    is_p = [False, False] + [True] * (n - 1)
    for i in range(2, math.isqrt(n) + 1):
        if is_p[i]:
            for k in range(i * i, n + 1, i):
                is_p[k] = False
    return [x for x in range(n + 1) if is_p[x]]

def mod3_arrangements_answer(n: int) -> int:
    sizes = Counter(x % 3 for x in range(1, n + 1))
    res = 1
    for size in sizes.values():
        res = res * fact_mod_answer(size) % MOD
    return res

assert prime_arrangements_answer(7) == 144 == prime_arrangements_brute(7)
assert [x for x in range(1, 20) if is_prime_answer(x)] == [2, 3, 5, 7, 11, 13, 17, 19]
assert fact_mod_answer(0) == 1 and fact_mod_answer(5) == 120
assert fact_mod_answer(100) == math.factorial(100) % MOD
assert prime_arrangements_answer(5) == 12
assert prime_arrangements_answer(100) == 682289015
assert primes_up_to_answer(1) == []
assert primes_up_to_answer(20) == [2, 3, 5, 7, 11, 13, 17, 19]
assert mod3_arrangements_answer(3) == 1
assert mod3_arrangements_answer(6) == 8
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


## 19. Template nhớ nhanh + checklist submit

```python
class Solution:
    def numPrimeArrangements(self, n: int) -> int:
        MOD = 10**9 + 7
        def is_prime(x):
            return x >= 2 and all(x % d for d in range(2, math.isqrt(x) + 1))
        p = sum(is_prime(x) for x in range(1, n + 1))
        res = 1
        for i in range(2, p + 1):     res = res * i % MOD
        for i in range(2, n - p + 1): res = res * i % MOD
        return res
```

**Checklist:**
- [ ] `1` **không** phải số nguyên tố?
- [ ] Thử chia tới `isqrt(x) + 1` (bao gồm căn)?
- [ ] Đáp án = `p! × (n − p)!`, lấy `% (10**9 + 7)`?
- [ ] Test `n = 1, 2, 3, 5, 100`?
- [ ] Đã so với brute force cho `n` nhỏ khi nghi ngờ công thức?